# Dataset examples: creating, validating, and processing conversation datasets

This notebook shows how `apertus-common` is used to **create, validate, and
process datasets** given in the typical messages formats: build typed
conversations and store them, load a corpus lazily, audit and normalize it,
and convert OpenAI-style chat data in and out.

The library supports two formats:

- **Native** (fully supported): the typed model's own JSON, schema-described
  and lossless when read with `normalize=False`. `to_json` / `from_json` are
  its harmony-style names, and a
  dataset file is simply one conversation JSON per line. `from_json` carries
  the ingest flags `normalize`, `check`, and `strict`, so loading, conformance
  reporting, and correction are one call.
- **OpenAI chat-completions** (compatibility, lossy): `from_openai` /
  `to_openai` map the `messages` list used by OpenAI-style APIs and many
  datasets. The adapter accepts the documented system, developer, user,
  assistant, tool, text, image, audio, and function-call shapes. Unsupported
  roles or content shapes raise `AdapterError`; the reverse covers only the
  typed subset with an OpenAI counterpart and refuses the rest explicitly.

Contents:

1. Build a small dataset and store it as JSON lines
2. Load and iterate lazily
3. Validate a dataset: model errors, conformance reports, strict ingest
4. Normalize and store the corrected dataset
5. OpenAI messages in: `from_openai`
6. OpenAI messages out: `to_openai`, and what is lost
7. A pipeline end to end, including Hugging Face `datasets` rows

Launch from the repository root with:

```sh
uv run --with jupyterlab jupyter lab examples/dataset_examples.ipynb
```

> The notebook imports directly from `src/`, so an editable package install is not required. Run cells from top to bottom.

In [ ]:
import sys
from pathlib import Path

repo_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").exists()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Open this notebook from inside the apertus-common repository")
src = str(repo_root / "src")
if src not in sys.path:
    sys.path.insert(0, src)
print(f"Using repository: {repo_root}")

In [ ]:
import json
import tempfile
from collections import Counter
from pprint import pprint
from typing import Any

from pydantic import ValidationError

from apertus_common import (
    AdapterError,
    Assistant,
    ConformanceError,
    Conversation,
    Effort,
    Harness,
    Retrieval,
    SystemPrompt,
    TextPart,
    Think,
    ToolCall,
    ToolResult,
    ToolSpec,
    User,
    Wait,
    from_json,
    from_openai,
    render_str,
    to_json,
    to_openai,
)

workdir = Path(tempfile.mkdtemp(prefix="apertus-dataset-"))
print(f"Working directory: {workdir}")

## 1. Build a small dataset and store it as JSON lines

The typed model's JSON is the one canonical serialization; `to_json` is its
harmony-style name (`Conversation.model_dump_json()`). A dataset file is the
JSON-lines convention over it: **one conversation JSON per line**. That is a
two-line recipe, not a library helper. `to_json` preserves the complete typed
value, and `from_json(line, normalize=False)` is its exact inverse. The default
`normalize=True` deliberately corrects input order while loading. The line
schema is `Conversation.model_json_schema()`, so external tools can validate
a corpus without importing the library.

In [ ]:
weather = ToolSpec(
    name="get_weather",
    json_schema={
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
    },
    policy="One city per call.",
)
system = SystemPrompt(identity="You are Aria.", effort=Effort.MEDIUM, tools=[weather])

dataset = [
    Conversation(
        system=system,
        items=[
            User(payload=[TextPart(text="What should I wear in Lisbon?")]),
            Think(payload="Need the weather first."),
            ToolCall(id="get_weather:0", name="get_weather", args={"city": "Lisbon"}),
            Wait(),
            ToolResult(id="get_weather:0", payload='{"tempC":19,"cond":"rain"}'),
            Assistant(payload="19C and rainy; bring a jacket."),
            Wait(),
        ],
    ),
    Conversation(
        system=system,
        items=[
            User(payload=[TextPart(text="Say hi.")]),
            Assistant(payload="Hi!"),
            Wait(),
        ],
    ),
    # deliberately non-canonical: the harness notice arrives after the user message
    Conversation(
        system=system,
        items=[
            User(payload=[TextPart(text="What time is it?")]),
            Harness(payload="Current time: 14:32."),
            Assistant(payload="It is 14:32."),
            Wait(),
        ],
    ),
]

dataset_path = workdir / "conversations.jsonl"
with dataset_path.open("w", encoding="utf-8") as sink:
    sink.writelines(to_json(conversation) + "\n" for conversation in dataset)

lines = dataset_path.read_text(encoding="utf-8").splitlines()
assert len(lines) == len(dataset)
assert lines[1] == dataset[1].model_dump_json()
print(lines[1][:120], "...")

## 2. Load and iterate lazily

`from_json` per line: an open file streams line by line, so a corpus never
has to fit in memory. By default `from_json` applies canonical input order
(`normalize=True`). Each conversation renders to the Apertus string form with
`render_str`.

In [ ]:
with dataset_path.open(encoding="utf-8") as source:
    conversations = (from_json(line) for line in source)
    for number, conversation in enumerate(conversations, start=1):
        print(f"--- conversation {number}: {len(conversation.items)} items")
        # print only the items; the shared system block is the same in every line
        print(render_str(conversation).split("<|/sys|>", 1)[1].strip())

## 3. Validate a dataset

Two independent layers apply. **Model validation** is pydantic's: a line
that is not a valid `Conversation` raises `ValidationError` from `from_json`;
the loop owns the line number. **Conformance** is the checker: cross-message
rules such as canonical input order, declared tools, and tool-result echoes.
`from_json(..., check=True)` returns `(conversation, report)`;
`strict=True` raises `ConformanceError` on error-severity findings (warnings
pass).

To audit a corpus without correcting it, load with `normalize=False` so the
observed input order is preserved and reported.

In [ ]:
def audit(path: Path) -> Counter[str]:
    """Count conformance violations by rule id over a JSON-lines corpus."""
    with path.open(encoding="utf-8") as source:
        return Counter(
            violation.rule
            for line in source
            for violation in from_json(line, normalize=False, check=True)[1]
        )


print(audit(dataset_path))

In [ ]:
corrupt_path = workdir / "corrupt.jsonl"
bad_line = '{"items": [{"direction": "in", "type": "user"}]}'
corrupt_path.write_text(
    "\n".join([lines[0], bad_line, lines[1]]) + "\n", encoding="utf-8"
)

with corrupt_path.open(encoding="utf-8") as source:
    for number, line in enumerate(source, start=1):
        try:
            from_json(line)
        except ValidationError as error:
            print(f"line {number}: {str(error).splitlines()[0]}")
            assert number == 2

A pipeline that wants to skip bad records keeps its own tally; and with
`strict=True` a conformance error is just as loud as a model error:

In [ ]:
def load_tolerant(
    path: Path, *, normalize: bool = True
) -> tuple[list[Conversation], dict[int, Exception]]:
    """Load every good line; collect model and conformance errors by line number."""
    good: list[Conversation] = []
    bad: dict[int, Exception] = {}
    with path.open(encoding="utf-8") as source:
        for number, line in enumerate(source, start=1):
            try:
                good.append(from_json(line, normalize=normalize, strict=True))
            except (ValidationError, ConformanceError) as error:
                bad[number] = error
    return good, bad


good, bad = load_tolerant(corrupt_path)
print(len(good), "valid; failed lines:", sorted(bad))
# without normalization the non-canonical third conversation fails strict ingest too
good, bad = load_tolerant(dataset_path, normalize=False)
print(
    len(good),
    "valid; failed lines:",
    {number: str(error)[:60] for number, error in bad.items()},
)

## 4. Normalize and store the corrected dataset

By default `from_json` applies `Conversation.normalized()`: each run of
consecutive input messages is stably sorted into canonical delivery order
(harness, tool result, retrieval/event, attachment, user). Outputs and waits
never move. Writing the normalized conversations back with `to_json` gives a
corpus that the checker accepts; the round trip through `from_json`/`to_json`
is then byte-stable.

In [ ]:
normalized_path = workdir / "conversations.normalized.jsonl"
with (
    dataset_path.open(encoding="utf-8") as source,
    normalized_path.open("w", encoding="utf-8") as sink,
):
    sink.writelines(to_json(from_json(line)) + "\n" for line in source)

print("before:", audit(dataset_path))
print("after: ", audit(normalized_path))

normalized_lines = normalized_path.read_text(encoding="utf-8").splitlines()
third = from_json(normalized_lines[2])
print([item.type for item in third.items])  # harness now precedes user
assert [
    to_json(from_json(line)) for line in normalized_lines
] == normalized_lines  # stable

## 5. OpenAI messages in: `from_openai`

Many datasets and every chat-completions request use the OpenAI message
list. `from_openai(messages, tools=..., effort=...)` lifts it into the typed
model:

- a leading `system`/`developer` message becomes the system prompt's
  `behavior` (there is always a system prompt: `effort` is required, pass
  the request's `reasoning_effort`); a later `system` becomes `Harness`,
  and the default `normalize=True` moves such a notice ahead of the `user`
  or `tool` messages it followed (canonical order; `normalize=False` keeps
  arrival order);
- `user` content maps to text, image (`image_url.url` as `ref`) and audio
  (`input_audio.data` as `ref`) parts;
- an `assistant` message yields, in order, `Think` from `reasoning_content`,
  `Assistant` from `content`, and one `ToolCall` per `tool_calls` entry
  (JSON-string arguments parsed, ids kept byte for byte);
- `tool` becomes `ToolResult`; `tools` become `ToolSpec` entries;
- a `Wait` closes each run of assistant messages. Pass
  `continue_final_message=True` to leave a trailing assistant run open
  (prefill; `check` reports the open run as a warning);
- the ingest flags are the same as for `from_json`: `normalize`, `check`,
  `strict`.

In [ ]:
tools: list[dict[str, Any]] = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "One city per call.",
            "parameters": weather.model_dump(mode="json")["json_schema"],
        },
    }
]
messages: list[dict[str, Any]] = [
    {"role": "system", "content": "Be brief."},
    {"role": "user", "content": "Lisbon or Porto?"},
    {
        "role": "assistant",
        "reasoning_content": "Fetch both.",
        "content": None,
        "tool_calls": [
            {
                "id": "call_abc123",
                "type": "function",
                "function": {"name": "get_weather", "arguments": '{"city":"Lisbon"}'},
            },
            {
                "id": "call_def456",
                "type": "function",
                "function": {"name": "get_weather", "arguments": '{"city":"Porto"}'},
            },
        ],
    },
    {"role": "tool", "tool_call_id": "call_abc123", "content": "24C, sunny"},
    {"role": "tool", "tool_call_id": "call_def456", "content": "19C, cloudy"},
    {"role": "assistant", "content": "Lisbon is warmer."},
]

lifted, report = from_openai(messages, tools, effort="high", check=True)
print([item.type for item in lifted.items])
assert report.ok
print(render_str(lifted))

In [ ]:
prefill, report = from_openai(
    [
        {"role": "user", "content": "Write a haiku."},
        {"role": "assistant", "content": "Autumn wind..."},
        {"role": "user", "content": "Another, starting with 'Winter'."},
        {"role": "assistant", "content": "Winter"},
    ],
    continue_final_message=True,
    check=True,
)
# no trailing wait: the last burst stays open, and the checker says so
print([item.type for item in prefill.items])
print([violation.rule for violation in report.warnings])

## 6. OpenAI messages out: `to_openai`, and what is lost

The OpenAI format has one fixed role set, so `to_openai` is a **lossy**
projection with a stated subset:

| direction | supported | not representable |
|---|---|---|
| in (`from_openai`) | `system`/`developer`, `user`, `assistant`, and `tool` messages with the documented text, image, audio, and function-call shapes | unsupported roles or content-part shapes raise `AdapterError`; accepted input round-trips only up to normalization: `developer` comes back as `system`, assistant `content` is always present (`null` when absent), `tool_calls` carry `type` and compact re-serialized `arguments`, `reasoning_effort` is emitted, optional detail fields and unknown keys are dropped, and default normalization may reorder a later system notice |
| out (`to_openai`) | `Harness` (as `system`), `User`, `ToolResult`, `Think`, `Assistant`, `ToolCall`, `Wait`, system prompt without `extra_tags` | `Retrieval`, `Event`, `Attachment`, `Document`, generic types, `VerifiableAnswer`, header `extras`, system `extra_tags`: **`AdapterError`**, never dropped or relabelled |

Two things are lossy rather than refused: identity, behavior and environment
collapse into one `system` string, and a burst whose order is not
"reasoning, content, calls" is split into several assistant messages so that
reasoning never enters `content`.

In [ ]:
request = to_openai(lifted)
assert request["messages"] == messages  # inside the subset: exact round trip
assert request["tools"] == tools
pprint(to_openai(dataset[0]), width=100)

In [ ]:
# an interleaved burst becomes two assistant messages; reasoning never enters content
interleaved = Conversation(
    items=[
        User(payload=[TextPart(text="hi")]),
        Assistant(payload="One moment."),
        Think(payload="Now reconsider."),
        Assistant(payload="Here you go."),
        Wait(),
    ]
)
pprint(to_openai(interleaved)["messages"], width=100)

In [ ]:
# items without an OpenAI counterpart raise instead of being relabelled
with_retrieval = Conversation(
    items=[Retrieval(src="docs:7", payload="..."), User(payload=[TextPart(text="hi")])]
)
try:
    to_openai(with_retrieval)
except AdapterError as error:
    print(error, "|", error.where, error.index)

## 7. A pipeline end to end

Ingest an OpenAI-style JSON-lines corpus (`{"messages": [...], "tools":
[...]}` per line), lift each record, and store the result as native JSON
lines. After this conversion, `check` and `render_str` work on typed
conversations; the planned tokenizer integration will consume the same model.

In [ ]:
openai_records = [
    {"messages": messages, "tools": tools},
    {
        "messages": [
            {"role": "user", "content": "ping"},
            {"role": "assistant", "content": "pong"},
        ]
    },
]
openai_path = workdir / "openai_corpus.jsonl"
openai_path.write_text(
    "".join(json.dumps(record) + "\n" for record in openai_records), encoding="utf-8"
)

# stream: read a record, lift it, write a line; nothing is held in memory
converted_path = workdir / "converted.jsonl"
with (
    openai_path.open(encoding="utf-8") as source,
    converted_path.open("w", encoding="utf-8") as sink,
):
    records = (json.loads(line) for line in source)
    sink.writelines(
        to_json(from_openai(r["messages"], r.get("tools"))) + "\n" for r in records
    )
print(audit(converted_path) or "no violations")

The same two calls work on a Hugging Face `datasets` row: chat datasets keep
their conversation in a `messages` column (a list of role dicts, exactly the
chat-completions shape) and sometimes a `tools` column (a list of dicts, or a
JSON string). Arrow fills absent keys with `None` (`tool_calls: None`,
`reasoning_content: None`, `name: None`), which the adapter treats as
absent. `datasets` is not a dependency of this library, so the snippet is
shown, not run:

```python
from datasets import load_dataset

rows = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True)

def to_conversation(row: dict[str, Any]) -> Conversation:
    tools = row.get("tools")
    if isinstance(tools, str):        # some datasets store tools as a JSON string
        tools = json.loads(tools)
    return from_openai(row["messages"], tools)

with open("ultrachat.apertus.jsonl", "w", encoding="utf-8") as sink:
    sink.writelines(to_json(to_conversation(row)) + "\n" for row in rows)
```

The next cell runs the same function on rows shaped like a `datasets`
batch, including Arrow's `None`-filled keys.

In [ ]:
def to_conversation(row: dict[str, Any]) -> Conversation:
    tools = row.get("tools")
    if isinstance(tools, str):
        tools = json.loads(tools)
    return from_openai(row["messages"], tools)


hf_like_rows: list[dict[str, Any]] = [
    {
        "prompt": "ping",
        "messages": [
            {"role": "user", "content": "ping", "tool_calls": None, "name": None},
            {"role": "assistant", "content": "pong", "tool_calls": None, "name": None},
        ],
        "tools": None,
    },
    {"messages": messages, "tools": json.dumps(tools)},
]
for row in hf_like_rows:
    print([item.type for item in to_conversation(row).items])

A corpus already in the native form travels through `datasets` as **text
lines**, not as nested columns: Arrow needs one type per column, and an
item `payload` is a string for most types but a part list for `user` and
`attachment`, so `load_dataset("json", ...)` cannot infer a schema for it.
Load the `.jsonl` with the `text` builder (one line per row) or keep the
conversation JSON in a string column, and hand the strings to `from_json`:

```python
rows = load_dataset("text", data_files="conversations.jsonl", split="train", streaming=True)
conversations = (from_json(row["text"]) for row in rows)          # lazy, normalized

# or a JSON-string column, e.g. {"id": ..., "conversation": "<one conversation JSON>"}
conversations = (from_json(row["conversation"]) for row in rows)
```

Going the other way, `to_json` lines are exactly what such a `text` or string
column holds.

In [ ]:
# rows as the "text" builder yields them: one conversation JSON per row
text_rows = [
    {"text": line} for line in dataset_path.read_text(encoding="utf-8").splitlines()
]
native = [from_json(row["text"]) for row in text_rows]
assert native == [conversation.normalized() for conversation in dataset]

# rows with the conversation JSON in a string column next to metadata
column_rows = [
    {"id": str(number), "conversation": to_json(conversation)}
    for number, conversation in enumerate(dataset)
]
assert [
    from_json(row["conversation"], normalize=False) for row in column_rows
] == dataset
print(len(native), "native conversations loaded from dataset rows")